In [ ]:
import numpy as np
from alc import diagRH
from alc import traspuesta
from alc import norma
from alc import inversa
from alc import producto_interno

In [66]:
def obtenerQ2d(imgPts,objPts):
  matriz_Q=[] #Armo mi matriz Q en donde voy a guardar las filas creadas

  for i in range(len(objPts)): #Separo las coordenadas y aplico el algoritmo
    U= objPts[i][0]
    V= objPts[i][1]
    u= imgPts[i][0]
    v= imgPts[i][1]

    fila_1=[U, V, 1, 0, 0, 0, -u*U, -u*V, -u]
    fila_2=[0, 0, 0, U, V, 1, -v*U, -v*V, -v]

    matriz_Q.append(fila_1)
    matriz_Q.append(fila_2)

  return np.array(matriz_Q)

In [67]:
def encontrar_vector_minimo(array):
    #Armo mi matriz Q*T.Q=A
    Q_final=traspuesta(array) @ array
    #Llamo a la funcion diagRH  para quedarme con el menor autovalor
    S,D =diagRH(Q_final)
    
    menor= D[0,0]
    indice=0
    for i in range(1,D.shape[0]):#Busco el menor autovalor en la diagonal 
        if D[i][i] < menor:
            menor=  D[i,i]
            indice=i
    vector_h= S[:,indice]
    return vector_h #No lo transformo asi puedo reutilizar esta funcion

In [68]:
def armarA(Hlst):
    filas_A = []

    for H in Hlst:
        # Primera columna de la homografía
        a, b, c = H[:, 0]  #selecciona todas  las filas agarra la primer columna
        # Segunda columna de la homografía
        d, e, f = H[:, 1] #selecciona todas las filas agarra la segunda columna

        #c1*TxBxc2
        fila_1 = [ 
            a*d,
            a*e + b*d,
            b*e,
            a*f + c*d,
            b*f + c*e,
            c*f
        ]
        #c1*TxBxc1-c2*TxBxc2=0
        #Ver demo despues pero se re entendio   
        fila_2 = [
            a*a - d*d,
            2*(a*b - d*e),
            b*b - e*e,
            2*(a*c - d*f), 
            2*(b*c - e*f),
            c*c - f*f
        ]
        filas_A.append(fila_1)
        filas_A.append(fila_2)
    return np.array(filas_A)

In [69]:
def normalizar_diagonal(B, autovalores, tol=1e-12):
    lista_positivo = []
    lista_negativo = []
    lista_cero = []

    for valor in autovalores:
        if valor > tol:
            lista_positivo.append(valor)
        elif valor < -tol:
            lista_negativo.append(valor)
        else:
            lista_cero.append(valor)

    if len(lista_cero) > 0:
        # raise lanza una excepción(un problema  ocurrido en la ejecucion)
        # valueerror tipo de exepcion
        raise ValueError("B tiene autovalores cercanos a cero")

    elif len(lista_positivo) > 0 and len(lista_negativo) > 0:
        raise ValueError("B es indefinida: no se puede aplicar Cholesky")

    elif len(lista_negativo) > 0:
        return -B

    else:
        return B

In [ ]:
def calcularK2D(A):
    b_final= encontrar_vector_minimo(A)
    B = np.array([
    [b_final[0], b_final[1], b_final[3]],
    [b_final[1], b_final[2], b_final[4]],
    [b_final[3], b_final[4], b_final[5]]])
    #Necesito aplicar cholesky para ello se que B  
    _, D =diagRH(B)

    lista_autovalores=[]
    for i in range(D.shape[0]):
        lista_autovalores.append(D[i][i])
    B=normalizar_diagonal(B,lista_autovalores)

    #Aca tengo que aplicar el algoritmo de cholesky (luego lo creo)
    L = np.linalg.cholesky(B)
    K  = traspuesta(inversa(L))
    if K[2][2]!=1:
        K=K/K[2][2]
    return K


In [ ]:
def obtenerPose2D(H, K):
    K_inv = inversa(K)
    c1 = H[:, 0]
    c2 = H[:, 1]
    c3 = H[:, 2]

    # Multiplicar por la inversa de K
    v1 = K_inv @ c1
    v2 = K_inv @ c2
    v3 = K_inv @ c3

    # Calcular el factor de escala
    if norma(v1) < 1e-12:
        raise ValueError("No se puede calcular lambda")
    
    lamda = 1 / norma(v1)
    if (lamda * v3)[2] < 0: #Chequeo que el valor de lamda sea positivo para  que  la camara apunta adelante
        lamda = -lamda
                
    # Recuperar r1, r2 y t
    r1 = lamda * v1
    r2 = lamda * v2
    t = lamda * v3

    r1 = r1 / norma(r1) #Normalizo ||r1||
    r2 = r2 - producto_interno(r1, r2) * r1   

    if norma(r2) < 1e-12:
        raise ValueError("No se puede construir la rotación")

    r2 = r2 / norma(r2) #Normalizo ||r2||

    #Armo ||r3|| como r1Xr2 ya que tiene que ser  ortogonal
    r3 = np.array([ 
        r1[1]*r2[2] - r1[2]*r2[1],
        r1[2]*r2[0] - r1[0]*r2[2],
        r1[0]*r2[1] - r1[1]*r2[0]])

    #Contruyo la matriz de rotación
    R = np.zeros((3, 3))

    for i in range(3):
        R[i, 0] = r1[i]
        R[i, 1] = r2[i]
        R[i, 2] = r3[i]

    return R, t

In [72]:
objPts1 = np.array([
    [0,   0],
    [60,  0],
    [180, 0],
    [270, 0],
    [0,  90],
    [120, 90],
    [270, 90],
    [0, 180]
])

In [73]:
# FOTOGRAFÍA 1
imgPts1 = np.array([
    [409.90, 352.18],
    [491.17, 353.19],
    [652.56, 354.90],
    [772.02, 355.87],
    [411.64, 228.58],
    [573.59, 230.75],
    [773.49, 233.64],
    [413.52, 105.19]
])

# FOTOGRAFÍA 2
imgPts2 = np.array([
    [482.43, 405.80],
    [561.17, 387.01],
    [717.60, 349.65],
    [833.06, 321.89],
    [454.77, 285.46],
    [611.99, 248.72],
    [805.97, 203.34],
    [427.18, 165.14]
])

# FOTOGRAFÍA 3
imgPts3 = np.array([
    [441.09, 386.07],
    [523.65, 380.74],
    [686.85, 370.41],
    [806.82, 362.59],
    [433.83, 260.08],
    [598.40, 250.80],
    [800.11, 239.01],
    [426.73, 134.05]
])

# FOTOGRAFÍA 4
imgPts4 = np.array([
    [451.53, 366.47],
    [533.50, 364.65],
    [695.81, 361.24],
    [815.26, 358.22],
    [448.98, 241.07],
    [613.83, 238.37],
    [816.12, 235.14],
    [446.48, 113.42]
])

# FOTOGRAFÍA 5
imgPts5 = np.array([
    [440.36, 331.18],
    [523.48, 336.77],
    [685.99, 347.27],
    [804.00, 355.00],
    [449.24, 204.50],
    [613.53, 217.40],
    [811.67, 232.73],
    [458.18,  78.65]
])

# FOTOGRAFÍA 6
imgPts6 = np.array([
    [444.85, 348.13],
    [527.71, 349.24],
    [690.13, 351.81],
    [808.40, 353.45],
    [447.52, 221.38],
    [611.86, 225.77],
    [810.62, 230.90],
    [450.30,  95.18]
])

# FOTOGRAFÍA 7
imgPts7 = np.array([
    [456.88, 396.94],
    [538.40, 393.41],
    [700.36, 386.80],
    [820.04, 381.37],
    [451.81, 272.38],
    [616.18, 266.28],
    [819.06, 258.59],
    [446.74, 145.62]
])

# FOTOGRAFÍA 8
imgPts8 = np.array([
    [452.69, 393.09],
    [537.44, 385.22],
    [700.43, 370.04],
    [816.56, 359.10],
    [442.79, 262.51],
    [610.02, 250.73],
    [806.89, 236.49],
    [432.92, 132.54]
])

# FOTOGRAFÍA 9
imgPts9 = np.array([
    [453.08, 393.46],
    [533.66, 377.53],
    [693.83, 346.02],
    [812.37, 322.41],
    [428.47, 270.60],
    [592.71, 238.99],
    [795.85, 199.83],
    [402.91, 142.56]
])

# FOTOGRAFÍA 10
imgPts10 = np.array([
    [441.82, 330.26],
    [528.47, 324.61],
    [699.92, 313.39],
    [826.02, 305.19],
    [435.36, 202.45],
    [603.38, 192.29],
    [809.96, 179.89],
    [429.41,  81.14]
])

In [74]:
imagenes = [
    imgPts1, imgPts2, imgPts3, imgPts4, imgPts5,
    imgPts6, imgPts7, imgPts8, imgPts9, imgPts10
]

lista_H = []

for puntos in imagenes:
    Q = obtenerQ2d(puntos,objPts1)
    h = encontrar_vector_minimo(Q)
    H = h.reshape(3, 3)
    lista_H.append(H)

A = armarA(lista_H)

print(A.shape)  # (20, 6)

K = calcularK2D(A)
print(K)

(20, 6)
Matriz B:
[[-1.13160074e-06 -1.15415528e-09  5.91474238e-04]
 [-1.15415528e-09 -1.09638237e-06  3.26187645e-04]
 [ 5.91474238e-04  3.26187645e-04 -9.99999772e-01]]
Autovalores de B:
[np.float64(-1.000000228119219), np.float64(-1.1040869652863168e-06), np.float64(-6.676555786060914e-07)]
[[724.60933033  -0.75082853 522.3852179 ]
 [  0.         736.15580939 296.96275711]
 [  0.           0.           1.        ]]
